In [1]:
import pandas as pd
import urllib.request
import os
from datetime import datetime
import glob

data_dir = "vhi_data"
if not os.path.exists(data_dir):
    os.makedirs(data_dir)

def download_vhi_data():
    for province_id in range(1, 28): 

        existing_files = glob.glob(f"{data_dir}/vhi_id_{province_id}_*.csv")
        
        if existing_files:
            print(f"Дані для області {province_id} вже завантажено: {existing_files[0]}")
            continue 

        url = f"https://www.star.nesdis.noaa.gov/smcd/emb/vci/VH/get_TS_admin.php?country=UKR&provinceID={province_id}&year1=1981&year2=2024&type=Mean"
        
        now = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
        filename = f"{data_dir}/vhi_id_{province_id}_{now}.csv"
        
        try:
            print(f"Завантаження даних для області {province_id}...")
            urllib.request.urlretrieve(url, filename)
            print(f"Успішно збережено як {filename}")
        except Exception as e:
            print(f"Помилка при завантаженні області {province_id}: {e}")

download_vhi_data()

Дані для області 1 вже завантажено: vhi_data/vhi_id_1_2026-03-12_23-12-15.csv
Дані для області 2 вже завантажено: vhi_data/vhi_id_2_2026-03-12_23-12-16.csv
Дані для області 3 вже завантажено: vhi_data/vhi_id_3_2026-03-12_23-12-17.csv
Дані для області 4 вже завантажено: vhi_data/vhi_id_4_2026-03-12_23-12-20.csv
Дані для області 5 вже завантажено: vhi_data/vhi_id_5_2026-03-12_23-12-22.csv
Дані для області 6 вже завантажено: vhi_data/vhi_id_6_2026-03-12_23-12-23.csv
Дані для області 7 вже завантажено: vhi_data/vhi_id_7_2026-03-12_23-12-24.csv
Дані для області 8 вже завантажено: vhi_data/vhi_id_8_2026-03-12_23-12-25.csv
Дані для області 9 вже завантажено: vhi_data/vhi_id_9_2026-03-12_23-12-26.csv
Дані для області 10 вже завантажено: vhi_data/vhi_id_10_2026-03-12_23-12-27.csv
Дані для області 11 вже завантажено: vhi_data/vhi_id_11_2026-03-12_23-12-28.csv
Дані для області 12 вже завантажено: vhi_data/vhi_id_12_2026-03-12_23-12-29.csv
Дані для області 13 вже завантажено: vhi_data/vhi_id_13_20

In [2]:
province_dict = {
    1: (22, 'Черкаська'), 2: (24, 'Чернігівська'), 3: (23, 'Чернівецька'),
    4: (25, 'Республіка Крим'), 5: (3, 'Дніпропетровська'), 6: (4, 'Донецька'),
    7: (8, 'Івано-Франківська'), 8: (19, 'Харківська'), 9: (20, 'Херсонська'),
    10: (21, 'Хмельницька'), 11: (9, 'Київська'), 12: (26, 'Київ'),
    13: (10, 'Кіровоградська'), 14: (11, 'Луганська'), 15: (12, 'Львівська'),
    16: (13, 'Миколаївська'), 17: (14, 'Одеська'), 18: (15, 'Полтавська'),
    19: (16, 'Рівненська'), 20: (27, 'Севастополь'), 21: (17, 'Сумська'),
    22: (18, 'Тернопільська'), 23: (6, 'Закарпатська'), 24: (1, 'Вінницька'),
    25: (2, 'Волинська'), 26: (7, 'Запорізька'), 27: (5, 'Житомирська')
}

def clean_and_merge_vhi_data(data_dir="vhi_data"):
    all_files = glob.glob(os.path.join(data_dir, "*.csv"))
    df_list = []

    for file in all_files:
        filename = os.path.basename(file)
        old_id = int(filename.split('_')[2])
        new_id, prov_name = province_dict[old_id]

        headers = ['Year', 'Week', 'SMN', 'SMT', 'VCI', 'TCI', 'VHI', 'empty']
        df = pd.read_csv(file, header=1, names=headers, skipinitialspace=True)
        
        df = df.drop(columns=['empty'], errors='ignore')

        df = df.dropna()
        df = df[df['Year'].astype(str).str.contains(r'^\d+$')]

        df = df.astype({
            'Year': int, 'Week': int, 'SMN': float, 
            'SMT': float, 'VCI': float, 'TCI': float, 'VHI': float
        })

        df = df[df['VHI'] != -1.0]

        df['Area_ID'] = new_id
        df['Area_Name'] = prov_name

        df_list.append(df)

    full_df = pd.concat(df_list, ignore_index=True)
    return full_df

vhi_df = clean_and_merge_vhi_data()

print("Дані успішно очищено та об'єднано!")
display(vhi_df.head())

Дані успішно очищено та об'єднано!


,Year,Week,SMN,SMT,VCI,TCI,VHI,Area_ID,Area_Name
0,1982,2,0.034,251.07,40.82,71.85,56.33,11,Луганська
1,1982,3,0.032,252.56,38.21,64.53,51.37,11,Луганська
2,1982,4,0.030,254.76,32.74,55.45,44.10,11,Луганська
3,1982,5,0.027,256.18,26.61,53.62,40.12,11,Луганська
4,1982,6,0.025,256.78,20.92,56.90,38.91,11,Луганська


### Завдання 1: Реалізувати процедуру для формування вибірок (ряд VHI за рік для вказаної області)

In [3]:
def get_vhi_series(df, province_id, year):
    result = df[(df['Area_ID'] == province_id) & (df['Year'] == year)][['Week', 'VHI']]
    return result

print("VHI для Вінницької області (ID 1) за 2020 рік:")
display(get_vhi_series(vhi_df, 1, 2020).head())

VHI для Вінницької області (ID 1) за 2020 рік:


,Week,VHI
54365,1,40.92
54366,2,43.19
54367,3,44.74
54368,4,45.29
54369,5,44.80


### Завдання 2: Реалізувати процедуру для формування вибірок (VHI за вказаний діапазон років для вказаних областей)

In [4]:
def get_vhi_range(df, province_ids, year_min, year_max):
    result = df[
        (df['Area_ID'].isin(province_ids)) &
        (df['Year'] >= year_min) &
        (df['Year'] <= year_max)
    ][['Year', 'Week', 'VHI', 'Area_Name']]
    return result

print("VHI для Вінницької (1) та Київської (9) областей за 2015-2017 роки:")
display(get_vhi_range(vhi_df, [1, 9], 2015, 2017).sample(10))

VHI для Вінницької (1) та Київської (9) областей за 2015-2017 роки:


,Year,Week,VHI,Area_Name
32406,2017,48,41.54,Київська
32314,2016,8,39.04,Київська
32294,2015,40,40.82,Київська
54237,2017,29,58.19,Вінницька
32284,2015,30,45.28,Київська
54109,2015,5,46.44,Вінницька
54201,2016,45,45.18,Вінницька
54205,2016,49,49.51,Вінницька
32349,2016,43,53.33,Київська
54170,2016,14,45.66,Вінницька


### Завдання 3: Реалізувати процедуру для пошуку екстремумів (min та max) та медіани за вказані роки для вказаних областей

In [5]:
def get_vhi_stats(df, province_ids, year_min, year_max):
    filtered_df = df[
        (df['Area_ID'].isin(province_ids)) &
        (df['Year'] >= year_min) &
        (df['Year'] <= year_max)
    ]
    
    stats = {
        'Min': filtered_df['VHI'].min(),
        'Max': filtered_df['VHI'].max(),
        'Mean': round(filtered_df['VHI'].mean(), 2),
        'Median': filtered_df['VHI'].median()
    }
    return stats

print("Статистика VHI для Дніпропетровської (3) області за 2000-2010 роки:")
print(get_vhi_stats(vhi_df, [3], 2000, 2010))

Статистика VHI для Дніпропетровської (3) області за 2000-2010 роки:
{'Min': np.float64(17.77), 'Max': np.float64(90.29), 'Mean': np.float64(48.59), 'Median': np.float64(47.295)}
